# KG Edge Merge — Phase 2 Completion

Merges the original unified KG edges with the new typed edges from:
- **Step 1**: Qualifier-based typing (`nlp_kg_edges_qualified.parquet`)
- **Step 2**: LLM relation extraction (`nlp_kg_edges_llm.parquet`)

The node table (`unified_kg_nodes.parquet`) is unchanged — only edges are updated.

Output: overwritten `unified_kg_edges.parquet` with all typed edges merged in.

In [1]:
import sys
from pathlib import Path

import pandas as pd

DATA       = Path("data")
UNIFIED    = DATA / "unified_kg"
NLP        = DATA / "nlp_kg"

EDGES_ORIG      = UNIFIED / "unified_kg_edges.parquet"
EDGES_QUALIFIED = NLP / "nlp_kg_edges_qualified.parquet"
EDGES_LLM       = NLP / "nlp_kg_edges_llm.parquet"
NODES_PATH      = UNIFIED / "unified_kg_nodes.parquet"
EDGES_OUT       = UNIFIED / "unified_kg_edges.parquet"   # overwrites in place

print("Paths OK")

Paths OK


In [2]:
# Load all three edge files
orig      = pd.read_parquet(EDGES_ORIG)
qualified = pd.read_parquet(EDGES_QUALIFIED)
llm       = pd.read_parquet(EDGES_LLM)

print(f"Original edges      : {len(orig):>10,}")
print(f"Qualifier edges     : {len(qualified):>10,}")
print(f"LLM edges           : {len(llm):>10,}")
print()
print("Original relation breakdown:")
print(orig["relation_type"].value_counts().to_string())

Original edges      :    329,566
Qualifier edges     : 14,790,106
LLM edges           :         52

Original relation breakdown:
relation_type
co_occurs_with              281629
narrower_term_of             42519
associated_with_disorder      2107
implicated_in                 1803
co_activates_with              931
expressed_in                   577


In [3]:
# Validate: both endpoints must exist in unified_kg_nodes
nodes     = pd.read_parquet(NODES_PATH)
valid_ids = set(nodes["canonical_id"])

def check_endpoints(df: pd.DataFrame, name: str) -> pd.DataFrame:
    mask = df["subject_id"].isin(valid_ids) & df["object_id"].isin(valid_ids)
    dropped = (~mask).sum()
    if dropped:
        print(f"  {name}: dropping {dropped:,} edges with unknown endpoints")
    return df[mask].reset_index(drop=True)

orig      = check_endpoints(orig,      "original")
qualified = check_endpoints(qualified, "qualified")
llm       = check_endpoints(llm,       "llm")

print("Endpoint validation complete.")

Endpoint validation complete.


In [4]:
# Concatenate all three sources
merged = pd.concat([orig, qualified, llm], ignore_index=True)
print(f"Total before dedup: {len(merged):,}")

# Deduplicate: same (subject, relation, object) may appear in multiple sources.
# Keep the row with the highest weight (most paper support); break ties by source_kg priority.
SOURCE_PRIORITY = {"mesh": 0, "mesh_qualifier": 1, "llm_re": 2, "cogatlas": 3, "nlp": 4}
merged["_src_rank"] = merged["source_kg"].map(SOURCE_PRIORITY).fillna(99)

merged = (
    merged
    .sort_values(["weight", "_src_rank"], ascending=[False, True])
    .drop_duplicates(subset=["subject_id", "relation_type", "object_id"], keep="first")
    .drop(columns=["_src_rank"])
    .reset_index(drop=True)
)

# Drop any self-loops that slipped through
merged = merged[merged["subject_id"] != merged["object_id"]].reset_index(drop=True)

print(f"Total after dedup   : {len(merged):,}")
print(f"Edges removed       : {len(orig) + len(qualified) + len(llm) - len(merged):,}")

Total before dedup: 15,119,724
Total after dedup   : 15,113,176
Edges removed       : 6,548


In [5]:
# Relation type breakdown
counts = merged["relation_type"].value_counts()
total  = len(merged)

print(f"{'relation_type':<30} {'count':>12}   {'%':>6}")
print("-" * 54)
for rel, cnt in counts.items():
    print(f"{rel:<30} {cnt:>12,}   {100*cnt/total:>5.1f}%")
print("-" * 54)
print(f"{'TOTAL':<30} {total:>12,}   100.0%")
print()
print(f"Source breakdown:")
print(merged["source_kg"].value_counts().to_string())

relation_type                         count        %
------------------------------------------------------
implicated_in                     6,619,101    43.8%
associated_with_disorder          4,185,364    27.7%
treated_by                        2,695,381    17.8%
used_in                           1,290,297     8.5%
co_occurs_with                      278,985     1.8%
narrower_term_of                     42,519     0.3%
co_activates_with                       933     0.0%
expressed_in                            596     0.0%
------------------------------------------------------
TOTAL                            15,113,176   100.0%

Source breakdown:
source_kg
mesh_qualifier    14786202
mesh                303310
nlp                  23612
llm_re                  52


In [6]:
# Spot-check: 3 examples per relation type with human-readable names
name_map = nodes.set_index("canonical_id")["name"].to_dict()

for rel in merged["relation_type"].unique():
    sample = merged[merged["relation_type"] == rel].head(3)
    print(f"--- {rel} ---")
    for _, r in sample.iterrows():
        s = name_map.get(r["subject_id"], r["subject_id"])
        o = name_map.get(r["object_id"],  r["object_id"])
        print(f"  {s}  →  {o}  (w={r['weight']:.0f}, src={r['source_kg']})")
    print()

--- co_occurs_with ---
  Brain  →  Humans  (w=158330, src=mesh)
  Brain  →  Magnetic Resonance Imaging  (w=103503, src=mesh)
  Brain  →  Male  (w=96730, src=mesh)

--- used_in ---
  Magnetic Resonance Imaging  →  Humans  (w=140462, src=mesh_qualifier)
  Humans  →  Magnetic Resonance Imaging  (w=140462, src=mesh_qualifier)
  Magnetic Resonance Imaging  →  Female  (w=79257, src=mesh_qualifier)

--- implicated_in ---
  Brain  →  Humans  (w=115363, src=mesh_qualifier)
  Humans  →  Brain  (w=115363, src=mesh_qualifier)
  Brain  →  Male  (w=79731, src=mesh_qualifier)

--- associated_with_disorder ---
  Brain  →  Humans  (w=56623, src=mesh_qualifier)
  Humans  →  Brain  (w=56623, src=mesh_qualifier)
  Brain  →  Magnetic Resonance Imaging  (w=40984, src=mesh_qualifier)

--- treated_by ---
  Brain Neoplasms  →  Humans  (w=12326, src=mesh_qualifier)
  Humans  →  Brain Neoplasms  (w=12326, src=mesh_qualifier)
  Brain Neoplasms  →  Magnetic Resonance Imaging  (w=8330, src=mesh_qualifier)

--- co_a

In [7]:
# Back up original before overwriting
backup = UNIFIED / "unified_kg_edges_backup.parquet"
if not backup.exists():
    orig_raw = pd.read_parquet(EDGES_ORIG)
    orig_raw.to_parquet(backup, index=False)
    print(f"Backup saved → {backup}")
else:
    print(f"Backup already exists at {backup} — skipping.")

merged.to_parquet(EDGES_OUT, index=False)
print(f"Saved {len(merged):,} edges → {EDGES_OUT}")

Backup saved → data/unified_kg/unified_kg_edges_backup.parquet
Saved 15,113,176 edges → data/unified_kg/unified_kg_edges.parquet


In [8]:
# Final summary: before vs after
old_counts = pd.read_parquet(backup)["relation_type"].value_counts()
new_counts = merged["relation_type"].value_counts()

all_types = sorted(set(old_counts.index) | set(new_counts.index))

print(f"{'relation_type':<30} {'before':>10} {'after':>12}")
print("-" * 56)
for rel in all_types:
    old = old_counts.get(rel, 0)
    new = new_counts.get(rel, 0)
    print(f"{rel:<30} {old:>10,} {new:>12,}")
print("-" * 56)
print(f"{'TOTAL':<30} {sum(old_counts):>10,} {len(merged):>12,}")

relation_type                      before        after
--------------------------------------------------------
associated_with_disorder            2,107    4,185,364
co_activates_with                     931          933
co_occurs_with                    281,629      278,985
expressed_in                          577          596
implicated_in                       1,803    6,619,101
narrower_term_of                   42,519       42,519
treated_by                              0    2,695,381
used_in                                 0    1,290,297
--------------------------------------------------------
TOTAL                             329,566   15,113,176
